# Plant Disease Classification Using Deep Learning and Transfer Learning

**CENG 476 - Introduction to Deep Learning**  
**Student:** Emir EVREN - **ID:** 210444038

This notebook is the **detailed, final, leakage-audited project notebook**. It contains the core code used in the project, the final ultra-strict split, model definitions, training logic, evaluation metrics, calibration, bootstrap confidence intervals, reproducibility checks, robustness tests, Grad-CAM references, and the external PlantDoc OOD test.

> **Important:** the original **99.76% ensemble** result is preserved only as a historical result that triggered the leakage audit. It is **not** the final benchmark.

### Final headline results

- **Custom CNN:** 84.62% accuracy, 0.7813 Macro-F1
- **ResNet18:** 97.66% accuracy, 0.9686 Macro-F1
- **EfficientNet-B0:** **99.01% accuracy, 0.9874 Macro-F1**
- **50/50 soft-voting ensemble:** **99.14% accuracy, 0.9897 Macro-F1**
- **3-seed EfficientNet mean:** **99.001% ± 0.229 percentage points**
- **Mapped PlantDoc OOD:** EfficientNet **23.31%**, ensemble **25.00%**

The cached outputs shown in this notebook come from the completed final runs. Re-running the cells uses the local artifacts under `outputs/` when available.


## 1. Environment and Project Paths

The notebook is written so that it can be opened from either the repository root or the `notebooks/` directory. Heavy model training is **not** automatically repeated. Analysis cells load the saved manifests, histories, metrics, and figures produced by the project scripts.


In [1]:
from pathlib import Path
import json
import sys
import numpy as np
import pandas as pd
import torch

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC = PROJECT_ROOT / "src"
OUTPUTS = PROJECT_ROOT / "outputs"
AUDIT = OUTPUTS / "audit"
FULL = AUDIT / "full_control"

if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print("Project root:", PROJECT_ROOT)
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Full-control folder exists:", FULL.exists())


Project root: <repository-root>
Python: 3.14.x
PyTorch: CUDA-enabled build used for final training
CUDA available: True on the training machine
Full-control folder exists: True


## 2. Final Ultra-Strict Dataset Protocol

The project originally used an image-level split. Because PlantVillage can contain multiple views of the same leaf, that protocol was audited for exact duplicates, perceptual near-duplicates, and mapped physical-leaf overlap.

The final benchmark uses the **ultra-strict manifest**:

- Train: **39,091**
- Validation: **4,462**
- Locked official test: **10,709**
- Total used: **54,262**
- Classes: **38**

The test set itself was not modified. Test images participated only in deterministic, model-independent integrity auditing and later post-hoc diagnostics; test labels and model predictions were not used to select quarantined train/validation samples.


In [2]:
ULTRA_MANIFEST = AUDIT / "official_leaf_safe_split_manifest_ultrastrict.csv"
ULTRA_SUMMARY = AUDIT / "official_leaf_safe_split_summary_ultrastrict.json"

manifest = pd.read_csv(ULTRA_MANIFEST, keep_default_na=False)
with ULTRA_SUMMARY.open("r", encoding="utf-8") as f:
    ultra_summary = json.load(f)

split_counts = manifest["split"].value_counts().reindex(["train", "validation", "test"])
print(split_counts)
print("Total:", len(manifest))
print("Classes:", manifest["class_index"].nunique())


split
train         39091
validation     4462
test          10709
Name: count, dtype: int64
Total: 54262
Classes: 38


### 2.1 Initial leakage audit versus final audit

The first split produced suspiciously strong results. The audit found:

- **10** exact cross-split duplicate pairs/groups
- **68** perceptual near-duplicate cross-split pairs under the original audit
- **4,956** mapped same-physical-leaf cross-split groups

The conservative official protocol then removed exact train/test collisions and respected mapped leaf identity. A final strict `dHash <= 4` review still found **39** cross-split pairs. To make the final experiment intentionally conservative, the lower-priority side of every one of those pairs was quarantined using the priority **test > validation > train**.

Result:

- Quarantined from train: **34**
- Quarantined from validation: **4**
- Quarantined from test: **0**
- Remaining strict `dHash <= 4` cross-split pairs: **0**


In [3]:
audit_before = pd.DataFrame({
    "check": [
        "Exact cross-split duplicate pairs/groups",
        "Perceptual near-duplicate pairs",
        "Mapped same-physical-leaf cross-split groups",
        "Final strict dHash<=4 cross-split pairs",
    ],
    "before": [10, 68, 4956, 39],
    "after_ultra": [0, 0, 0, 0],
})
print(audit_before.to_string(index=False))


                                      check  before  after_ultra
      Exact cross-split duplicate pairs/groups      10            0
              Perceptual near-duplicate pairs      68            0
 Mapped same-physical-leaf cross-split groups    4956            0
      Final strict dHash<=4 cross-split pairs      39            0


### 2.2 Important limitation of the split audit

The available physical-leaf mapping does not cover every PlantVillage image. Therefore the claim is **not** that mathematical uniqueness of every physical specimen has been proven. The final statement is narrower:

> No detected exact, mapped-leaf, or strict `dHash <= 4` cross-split overlap remains under the implemented audit protocol.

This distinction is important when interpreting the final near-99% result.


## 3. Image Preprocessing and Data Augmentation

Training augmentation is intentionally moderate. Validation and test preprocessing are deterministic.

### Training transform

`RandomResizedCrop(224, scale=(0.80,1.00))`  
`RandomHorizontalFlip(0.5)`  
`RandomRotation(±15°)`  
`ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2)`  
ImageNet normalization

### Validation / test transform

`Resize(256)` → `CenterCrop(224)` → tensor → ImageNet normalization.


In [4]:
from torchvision import transforms
from torchvision.transforms import InterpolationMode

IMAGE_SIZE = 224
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.80, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(
        degrees=15,
        interpolation=InterpolationMode.BILINEAR,
        fill=(128, 128, 128),
    ),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

evaluation_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

print(train_transform)
print()
print(evaluation_transform)


Compose(
    RandomResizedCrop(size=(224, 224), scale=(0.8, 1.0))
    RandomHorizontalFlip(p=0.5)
    RandomRotation(degrees=[-15.0, 15.0])
    ColorJitter(brightness=(0.8, 1.2), contrast=(0.8, 1.2), saturation=(0.8, 1.2), hue=None)
    ToTensor()
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
)

Compose(
    Resize(size=256)
    CenterCrop(size=(224, 224))
    ToTensor()
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
)


## 4. Model 1 — Custom CNN Trained From Scratch

The baseline network is intentionally small and is trained from random initialization. It uses four convolutional blocks followed by adaptive average pooling and a linear 38-class classifier.

The model outputs **raw logits**. There is **no Softmax layer before `CrossEntropyLoss`** because PyTorch's `CrossEntropyLoss` internally applies the required log-softmax operation.


In [5]:
from torch import nn

class BaselineCNN(nn.Module):
    def __init__(self, num_classes=38, dropout_rate=0.4):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )

        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(dropout_rate),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        return self.classifier(x)

baseline = BaselineCNN()
baseline_params = sum(p.numel() for p in baseline.parameters())
print("Output classes:", baseline.classifier[-1].out_features)
print("Trainable parameters:", f"{baseline_params:,}")


Output classes: 38
Trainable parameters: 399,142


## 5. Transfer Learning Models

Two ImageNet-pretrained backbones are fine-tuned end-to-end. Only the final classifier is replaced; the pretrained feature extractor remains trainable.

### ResNet18

`Dropout(0.30) + Linear(512, 38)`

### EfficientNet-B0

`Dropout(0.30) + Linear(1280, 38)`


In [6]:
from torchvision.models import (
    ResNet18_Weights, resnet18,
    EfficientNet_B0_Weights, efficientnet_b0,
)

def create_resnet18_transfer(num_classes=38, dropout_rate=0.3):
    model = resnet18(weights=ResNet18_Weights.DEFAULT)
    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(dropout_rate),
        nn.Linear(in_features, num_classes),
    )
    return model

def create_efficientnet_b0_transfer(num_classes=38, dropout_rate=0.3):
    model = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(dropout_rate, inplace=True),
        nn.Linear(in_features, num_classes),
    )
    return model

# Parameter counts from the final project configuration.
model_sizes = pd.DataFrame([
    ["Custom CNN", 399_142],
    ["ResNet18", 11_196_006],
    ["EfficientNet-B0", 4_056_226],
], columns=["Model", "Trainable parameters"])

print(model_sizes.to_string(index=False))


          Model  Trainable parameters
     Custom CNN               399142
       ResNet18             11196006
EfficientNet-B0              4056226


## 6. Loss Function, Optimizer, Learning Rates, and Regularization

All models use unweighted `CrossEntropyLoss`.

**Baseline CNN**

- Batch size: 64
- Max epochs: 15
- Learning rate: `5e-4`
- Dropout: 0.40
- Weight decay: `1e-4`
- Early-stopping patience: 6

**Transfer models**

- Batch size: 32
- Max epochs: 12
- Backbone learning rate: `1e-4`
- Classifier learning rate: `5e-4`
- Dropout: 0.30
- Weight decay: `1e-4`
- Early-stopping patience: 5

Optimizer: **AdamW**, betas `(0.9, 0.999)`  
Scheduler: **ReduceLROnPlateau**, monitor validation loss, factor `0.5`, patience `2`, minimum LR `1e-6`  
Checkpoint selection: **validation Macro-F1**, with validation loss as tie-breaker.


In [7]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau

# Example: EfficientNet-B0 differential learning rates
# backbone_parameters, classifier_parameters are split by parameter name.

criterion = nn.CrossEntropyLoss()

# optimizer = AdamW(
#     [
#         {"params": backbone_parameters, "lr": 1e-4, "name": "backbone"},
#         {"params": classifier_parameters, "lr": 5e-4, "name": "classifier"},
#     ],
#     betas=(0.9, 0.999),
#     weight_decay=1e-4,
# )
#
# scheduler = ReduceLROnPlateau(
#     optimizer,
#     mode="min",
#     factor=0.5,
#     patience=2,
#     min_lr=1e-6,
# )

config_table = pd.DataFrame([
    ["Baseline CNN", 64, 15, "5e-4", "5e-4", 0.40, "1e-4"],
    ["ResNet18", 32, 12, "1e-4", "5e-4", 0.30, "1e-4"],
    ["EfficientNet-B0", 32, 12, "1e-4", "5e-4", 0.30, "1e-4"],
], columns=["Model","Batch","Epochs","Backbone LR","Classifier LR","Dropout","Weight decay"])

print(config_table.to_string(index=False))


          Model  Batch  Epochs Backbone LR Classifier LR  Dropout Weight decay
     Baseline CNN     64      15        5e-4          5e-4      0.4        1e-4
       ResNet18     32      12        1e-4          5e-4      0.3        1e-4
EfficientNet-B0     32      12        1e-4          5e-4      0.3        1e-4


## 7. Core Training Loop

The actual project loop uses mixed precision on CUDA, tracks augmented-train, deterministic clean-train, and validation metrics every epoch, and saves the best model by validation Macro-F1.

The simplified code below mirrors the important logic.


In [8]:
def run_epoch(model, loader, criterion, device, optimizer=None, scaler=None):
    training = optimizer is not None
    model.train(training)

    total_loss = 0.0
    correct = 0
    n = 0
    all_targets, all_preds = [], []

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        if training:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(training):
            with torch.autocast(
                device_type=device.type,
                dtype=torch.float16,
                enabled=(device.type == "cuda"),
            ):
                logits = model(images)
                loss = criterion(logits, labels)

            if training:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

        preds = logits.argmax(dim=1)
        total_loss += loss.item() * labels.size(0)
        correct += (preds == labels).sum().item()
        n += labels.size(0)
        all_targets.extend(labels.cpu().tolist())
        all_preds.extend(preds.cpu().tolist())

    return {
        "loss": total_loss / n,
        "accuracy": correct / n,
        "targets": all_targets,
        "predictions": all_preds,
    }

print("Training loop tracks: loss, accuracy, predictions, Macro-F1")
print("AMP on CUDA: enabled")
print("Checkpoint criterion: validation Macro-F1")
print("Test loader used during training: NO")


Training loop tracks: loss, accuracy, predictions, Macro-F1
AMP on CUDA: enabled
Checkpoint criterion: validation Macro-F1
Test loader used during training: NO


## 8. Why Macro-F1 Is Used

Accuracy can be dominated by larger classes. Macro-F1 computes the F1 score independently for each of the 38 classes and then gives every class equal weight.

For checkpoint selection:

```python
f1_score(y_true, y_pred, average="macro", zero_division=0)
```

This is why a model is not selected only because it has high overall accuracy.


## 9. Best Checkpoints and Same-Domain Generalization

A deterministic clean-training subset of **20 samples per class = 760 images** is evaluated after every epoch. This is more interpretable than comparing validation against augmented training batches.

The transfer models show much smaller same-domain gaps than the baseline CNN.


In [9]:
gap = pd.DataFrame([
    ["ResNet18", 98.16, 98.81, 97.66],
    ["EfficientNet-B0", 99.74, 98.68, 99.01],
], columns=["Model", "Clean-train accuracy", "Validation accuracy", "Test accuracy"])

gap["CleanTrain-Val gap (pp)"] = gap["Clean-train accuracy"] - gap["Validation accuracy"]
gap["Val-Test gap (pp)"] = gap["Validation accuracy"] - gap["Test accuracy"]

print(gap.round(2).to_string(index=False))


          Model  Clean-train accuracy  Validation accuracy  Test accuracy  CleanTrain-Val gap (pp)  Val-Test gap (pp)
       ResNet18                 98.16                98.81          97.66                    -0.65               1.15
EfficientNet-B0                 99.74                98.68          99.01                     1.06              -0.33


## 10. Final Ultra-Strict Test Results

The following table is the primary final benchmark. These are the results to report instead of the historical image-level split.


In [10]:
final_results = pd.DataFrame([
    ["Custom CNN", 399_142, 84.620413, 0.8635, 0.7666, 0.781323, 0.836106, 0.995589, 1647],
    ["ResNet18", 11_196_006, 97.656177, 0.9734, 0.9676, 0.968603, 0.976314, 0.999929, 251],
    ["EfficientNet-B0", 4_056_226, 99.010178, 0.9897, 0.9855, 0.987373, 0.990114, 0.999955, 106],
    ["50/50 Ensemble", np.nan, 99.140910, 0.9908, 0.9891, 0.989733, 0.991396, 0.999976, 92],
], columns=[
    "Model","Parameters","Accuracy (%)","Macro Precision","Macro Recall",
    "Macro-F1","Weighted-F1","Macro AUC","Errors"
])

print(final_results.to_string(index=False))


          Model  Parameters  Accuracy (%)  Macro Precision  Macro Recall  Macro-F1  Weighted-F1  Macro AUC  Errors
     Custom CNN    399142.0     84.620413           0.8635        0.7666  0.781323     0.836106   0.995589    1647
       ResNet18  11196006.0     97.656177           0.9734        0.9676  0.968603     0.976314   0.999929     251
EfficientNet-B0   4056226.0     99.010178           0.9897        0.9855  0.987373     0.990114   0.999955     106
 50/50 Ensemble         NaN     99.140910           0.9908        0.9891  0.989733     0.991396   0.999976      92


### 10.1 Top-3 accuracy

The final tests also produced very high Top-3 accuracy:

- Baseline CNN: **95.48%**
- ResNet18: **99.79%**
- EfficientNet-B0: **99.91%**
- Ensemble: **99.95%**


## 11. Ensemble Weight Selection — Validation Only

ResNet18 and EfficientNet-B0 probabilities are combined by weighted soft voting.

The candidate weights were evaluated on the **validation set only**. The final 50/50 weight was selected because it gave the highest validation Macro-F1 among the predefined candidates.


In [11]:
ensemble_candidates = pd.DataFrame([
    ["100% ResNet18", 1.00, 0.00, 0.988122, 0.986237],
    ["75% ResNet / 25% Eff", 0.75, 0.25, 0.990587, 0.988722],
    ["50% / 50%", 0.50, 0.50, 0.993725, 0.992033],
    ["25% ResNet / 75% Eff", 0.25, 0.75, 0.989691, 0.986400],
    ["100% EfficientNet", 0.00, 1.00, 0.986777, 0.982431],
], columns=["Candidate","ResNet weight","EffNet weight","Val Accuracy","Val Macro-F1"])

print(ensemble_candidates.to_string(index=False))
print("\nSelected: 50% ResNet18 + 50% EfficientNet-B0")


              Candidate  ResNet weight  EffNet weight  Val Accuracy  Val Macro-F1
          100% ResNet18           1.00           0.00      0.988122      0.986237
75% ResNet / 25% Eff           0.75           0.25      0.990587      0.988722
              50% / 50%           0.50           0.50      0.993725      0.992033
25% ResNet / 75% Eff           0.25           0.75      0.989691      0.986400
      100% EfficientNet           0.00           1.00      0.986777      0.982431

Selected: 50% ResNet18 + 50% EfficientNet-B0


## 12. Calibration and Bootstrap Confidence Intervals

High accuracy alone is not enough. Calibration checks whether predicted confidence is aligned with empirical correctness.

Metrics used:

- **ECE** — Expected Calibration Error
- **NLL** — Negative Log-Likelihood
- **Multiclass Brier score**
- **1000-sample bootstrap 95% confidence intervals**

The low ECE values indicate strong same-domain calibration.


In [12]:
calibration = pd.DataFrame([
    ["EfficientNet-B0", 99.010178, 0.003845, "98.81–99.20"],
    ["50/50 Ensemble", 99.140910, 0.009121, "98.95–99.31"],
], columns=["Model","Accuracy (%)","ECE","Bootstrap 95% Accuracy CI"])

print(calibration.to_string(index=False))


          Model  Accuracy (%)      ECE Bootstrap 95% Accuracy CI
EfficientNet-B0     99.010178 0.003845               98.81–99.20
 50/50 Ensemble     99.140910 0.009121               98.95–99.31


If the full-control artifacts are present locally, the complete calibration tables can be loaded directly:


In [ ]:
for filename in [
    "metrics_calibration_summary.csv",
    "bootstrap_95ci.csv",
    "efficientnet_calibration_bins.csv",
    "ensemble_calibration_bins.csv",
]:
    path = FULL / filename
    print("\n==", filename, "==")
    if path.exists():
        display(pd.read_csv(path).head(20))
    else:
        print("Artifact not found locally:", path)


## 13. Random-Label Sanity Check

A useful leakage sanity test is to destroy the relationship between images and training labels.

Protocol:

- Balanced ultra-strict training subset: **20 images/class = 760**
- Balanced validation subset: **10 images/class = 380**
- Training labels are randomly shuffled
- Validation labels remain correct
- Pretrained EfficientNet backbone is frozen; classifier is trained for 5 epochs
- Chance accuracy for 38 classes: **1/38 ≈ 2.63%**

If the pipeline were leaking class information through filenames, split construction, or another hidden channel, validation performance could remain unexpectedly high. It did not.


In [13]:
random_label = pd.DataFrame([
    ["Chance level", 2.63, np.nan],
    ["Random-label validation", 1.84, 0.0183],
], columns=["Condition","Accuracy (%)","Macro-F1"])

print(random_label.to_string(index=False))
print("\nRandom-label sanity result: PASS")


                Condition  Accuracy (%)  Macro-F1
             Chance level          2.63       NaN
Random-label validation          1.84    0.0183

Random-label sanity result: PASS


## 14. Robustness / Corruption Stress Testing

The locked same-domain test images were also used for **post-hoc stress diagnostics only**. These tests were not used to tune the model.

Stress conditions include:

- brightness decrease / increase
- contrast reduction
- Gaussian blur
- JPEG quality reduction
- 15° rotation
- large center occlusion
- large border occlusion

Brightness, contrast, JPEG compression, and moderate rotation remained near the high-98% to 99% region. The strongest degradation among standard corruptions came from Gaussian blur.


In [14]:
robustness_highlights = pd.DataFrame([
    ["EfficientNet-B0", "Clean", 99.01],
    ["50/50 Ensemble", "Clean", 99.14],
    ["EfficientNet-B0", "Gaussian blur r=2", 84.08],
    ["50/50 Ensemble", "Gaussian blur r=2", 86.53],
], columns=["Model","Condition","Accuracy (%)"])

print(robustness_highlights.to_string(index=False))


          Model          Condition  Accuracy (%)
EfficientNet-B0              Clean         99.01
 50/50 Ensemble              Clean         99.14
EfficientNet-B0 Gaussian blur r=2         84.08
 50/50 Ensemble Gaussian blur r=2         86.53


In [ ]:
robustness_path = FULL / "robustness_stress.csv"
if robustness_path.exists():
    robustness = pd.read_csv(robustness_path)
    display(robustness)
else:
    print("Run run_full_control_all.bat to regenerate robustness_stress.csv")


## 15. Seed Stability / Reproducibility

The EfficientNet experiment was repeated using a **fixed ultra-strict manifest** with three predeclared seeds. The test results from **all** seeds are reported; the best seed is not selected as the benchmark.

This avoids presenting seed 777 simply because it happened to be the strongest run.


In [15]:
seed_results = pd.DataFrame([
    [42,  0.982431, 99.010178, 0.987373, 106],
    [123, 0.986704, 98.767392, 0.983668, 132],
    [777, 0.992056, 99.224951, 0.990149, 83],
], columns=["Seed","Validation Macro-F1","Test accuracy (%)","Test Macro-F1","Errors"])

print(seed_results.to_string(index=False))
print()
print("Mean accuracy: 99.001%")
print("Accuracy std: 0.229 percentage points")
print("Accuracy range: 0.458 percentage points")


 Seed  Validation Macro-F1  Test accuracy (%)  Test Macro-F1  Errors
   42             0.982431          99.010178       0.987373     106
  123             0.986704          98.767392       0.983668     132
  777             0.992056          99.224951       0.990149      83

Mean accuracy: 99.001%
Accuracy std: 0.229 percentage points
Accuracy range: 0.458 percentage points


## 16. External Out-of-Domain Evaluation — PlantDoc

PlantVillage is a controlled-domain dataset. To test whether the strong same-domain score transfers to less controlled images, the final models were evaluated on a manually mapped subset of the official **PlantDoc** test set.

- Evaluated images: **236**
- Source folders mapped: **27**
- No PlantDoc fine-tuning
- No PlantDoc hyperparameter tuning
- Same trained EfficientNet and ensemble

Because PlantDoc and PlantVillage do not have identical ontologies or acquisition conditions, this is an **OOD/domain-shift probe**, not a directly comparable benchmark.


In [16]:
plantdoc = pd.DataFrame([
    ["EfficientNet-B0", 23.31, 0.2183],
    ["50/50 Ensemble", 25.00, 0.2349],
], columns=["Model","PlantDoc accuracy (%)","Mapped Macro-F1"])

print(plantdoc.to_string(index=False))


          Model  PlantDoc accuracy (%)  Mapped Macro-F1
EfficientNet-B0                  23.31           0.2183
 50/50 Ensemble                  25.00           0.2349


### Interpretation

The large drop from ~99% on PlantVillage to ~23–25% on mapped PlantDoc is the strongest evidence that the model is **domain dependent**.

It does **not** mean the PlantVillage benchmark is fake. Instead:

- the final PlantVillage result is reproducible under the implemented leakage controls,
- severe conventional train overfitting is not strongly supported by the internal gaps,
- but the learned decision function does not transfer well to a substantially different image domain.

Therefore this system is **not validated for real-field deployment**.


## 17. Error Analysis and Grad-CAM

The full-control pipeline saves:

- per-class metrics
- confusion matrices
- all predictions
- wrong predictions
- high-confidence error contact sheets
- Grad-CAM contact sheets for correct and incorrect EfficientNet cases

Grad-CAM is treated as **supportive qualitative evidence**, not causal proof that the model uses only disease lesions.


In [17]:
figure_files = [
    FULL / "efficientnet_predictions.csv",
    FULL / "efficientnet_errors.csv",
    FULL / "efficientnet_per_class.csv",
]

for p in figure_files:
    print(p.name, "->", "FOUND" if p.exists() else "not found")


efficientnet_predictions.csv -> FOUND
efficientnet_errors.csv -> FOUND
efficientnet_per_class.csv -> FOUND


### Saved qualitative figures

![EfficientNet Grad-CAM — correct predictions](../outputs/figures/full_control/efficientnet_gradcam_correct.jpg)

![EfficientNet Grad-CAM — error cases](../outputs/figures/full_control/efficientnet_gradcam_errors.jpg)

![EfficientNet reliability diagram](../outputs/figures/full_control/efficientnet_reliability.png)

![EfficientNet robustness stress](../outputs/figures/full_control/efficientnet_robustness_stress.png)


## 18. Load Complete Full-Control Tables

The following cell makes the notebook useful as a compact analysis dashboard after cloning/running the project. It loads the detailed CSVs if they exist.


In [ ]:
artifact_names = [
    "generalization_gap.csv",
    "metrics_calibration_summary.csv",
    "bootstrap_95ci.csv",
    "efficientnet_seed_stability_runs.csv",
    "robustness_stress.csv",
    "shortcut_occlusion_stress.csv",
    "plantdoc_efficientnet_per_class.csv",
    "plantdoc_ensemble_per_class.csv",
]

for name in artifact_names:
    path = FULL / name
    print("\n" + "="*80)
    print(name)
    print("="*80)
    if path.exists():
        display(pd.read_csv(path).head(50))
    else:
        print("Missing:", path)


## 19. Final Scientific Interpretation

The full sequence of the project is:

1. Train a from-scratch CNN and transfer-learning models.
2. Observe an unusually strong image-level result.
3. Audit the split instead of accepting the number blindly.
4. Detect exact duplicates, perceptual near-duplicates, and same-leaf overlap.
5. Build a conservative official leaf-safe protocol.
6. Apply an additional ultra-strict quarantine to every remaining `dHash <= 4` cross-split pair.
7. Retrain all final models from the cleaned manifest.
8. Select checkpoints only from validation Macro-F1.
9. Select ensemble weights only from validation.
10. Evaluate the locked official test.
11. Add calibration, bootstrap CI, error analysis, random-label sanity, robustness, Grad-CAM, and seed stability.
12. Run an external PlantDoc OOD probe.

### Final claim

> **Near-99% PlantVillage performance is reproducible under the implemented leakage-audited protocol, but the large PlantDoc performance drop shows strong domain dependence and limited real-world generalization.**

### What should *not* be claimed

- Do not claim the model has 99% real-world plant-disease accuracy.
- Do not claim every unmapped physical PlantVillage leaf has been mathematically proven unique.
- Do not claim the entire difference from the old 99.76% result was caused only by leakage.
- Do not select the best seed after seeing test performance.


## 20. Reproduction Commands

From the repository root:

```bat
REM Build and run the final ultra-strict benchmark
call .\run_ultrastrict_all.bat

REM Calibration, bootstrap, error audit, robustness,
REM random-label sanity, Grad-CAM, PlantDoc OOD, and seed stability
call .\run_full_control_all.bat
```

The large training artifacts and model checkpoints are generated locally. The notebook is designed to show the final methodology and to reload the saved CSV/JSON evidence without automatically retraining the networks.
